# P05A — Generator data sintesis untuk evaluasi model

Jalankan notebook ini lebih dahulu. Ia membuat data kurva beban–lendutan berulang per balok, label prioritas jembatan sintesis, dan fitur perkerasan dengan label pembangkit untuk audit setelah clustering. Seluruh nilai berasal dari aturan simulasi, bukan pengukuran lapangan.

In [ ]:
# Folder kerja: jalankan dari folder mana pun. Sel ini pindah ke akar repositori agar jalur data relatif (praktikum/...) benar.
import os
from pathlib import Path
_akar = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "praktikum").is_dir() and (p / "notebooks").is_dir()), None)
if _akar is not None:
    os.chdir(_akar)
else:
    print("Akar repositori tidak ditemukan. Buka notebook dari dalam folder repositori hasil clone.")

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.datasets import make_blobs


In [2]:
SEED = 51505
rng = np.random.default_rng(SEED)
# Dari akar repositori: tulis ke praktikum/bab05/data_bab5; di Colab atau folder lain: data_bab5 di direktori kerja.
out_dir = Path("praktikum/bab05/data_bab5") if Path("praktikum/bab05").is_dir() else Path("data_bab5")
out_dir.mkdir(parents=True, exist_ok=True)


## 1. Kurva beban–lendutan dengan pengamatan berkelompok
Setiap balok memiliki beberapa tahap pembebanan; `beam_id` dipakai untuk menahan balok utuh saat latihan evaluasi.

In [3]:
n_beams, n_steps = 72, 18
loads = np.linspace(10, 180, n_steps)
rows = []
for b in range(n_beams):
    span = rng.uniform(4.0, 8.0)
    rho = rng.uniform(0.7, 1.8)
    temp = rng.uniform(20, 90)
    beam_factor = rng.normal(1.0, 0.06)
    for load in loads:
        thermal = 0.035 * (temp - 20)
        y = beam_factor * (0.17 * load * (span/6)**3 + 0.0027*load**2) / rho + thermal
        y += rng.normal(0, 3.0)
        rows.append((f"B{b+1:03d}", span, rho, temp, load, max(0.5, y)))
beam = pd.DataFrame(rows, columns=["beam_id", "span_m", "reinforcement_ratio", "temperature_C", "load_kN", "deflection_mm"])
beam.to_csv(out_dir / "data_sintesis_lendutan.csv", index=False)
# Satu baris per balok untuk evaluasi lintas balok pada workflow Orange.
beam_orange = (beam.sort_values("load_kN").groupby("beam_id", as_index=False).tail(1)[["beam_id", "span_m", "reinforcement_ratio", "temperature_C", "deflection_mm"]].sort_values("beam_id"))
beam_orange.to_csv(out_dir / "data_sintesis_lendutan_per_balok.csv", index=False)
print("Lendutan kurva:", beam.shape, "balok:", beam.beam_id.nunique())
print("Lendutan satu baris per balok:", beam_orange.shape)


Lendutan kurva: (1296, 6) balok: 72
Lendutan satu baris per balok: (72, 5)


## 2. Label sintesis prioritas inspeksi
Label digunakan untuk memperagakan metrik klasifikasi. Skor dan ambang tidak merepresentasikan aturan inspeksi aset.

In [4]:
m = 1200
bridge_id = np.array([f"J{i:04d}" for i in range(m)])
age = rng.integers(1, 101, m)
traffic = rng.integers(100, 15000, m)
scour = rng.uniform(0, 1, m)
deterioration = rng.uniform(0, 1, m)
flood = rng.uniform(0, 1, m)
latent = (-3.6 + .011*age + .0011*traffic + 2.6*scour + 2.4*deterioration + 1.5*flood + rng.normal(0, 1.0, m))
label = np.where(latent > 0.25, "prioritas", "rutin")
bridge = pd.DataFrame({"bridge_id": bridge_id, "age_years": age, "heavy_traffic_daily": traffic, "scour_index": scour.round(3), "deterioration_index": deterioration.round(3), "flood_exposure": flood.round(3), "priority_label": label})
bridge.to_csv(out_dir / "data_sintesis_prioritas.csv", index=False)
print("Prioritas:", bridge.priority_label.value_counts().to_dict())


Prioritas: {'prioritas': 1150, 'rutin': 50}


## 3. Fitur perkerasan dan label generator
Label pembangkit hanya untuk pemeriksaan eksternal setelah clustering. Jangan masukkan ID atau label sebagai prediktor clustering.

In [5]:
X, z = make_blobs(n_samples=700, centers=4, n_features=4, cluster_std=[0.8, 1.0, 1.1, 0.9], random_state=SEED)
cluster_data = pd.DataFrame(X, columns=["feature_1", "feature_2", "feature_3", "feature_4"])
cluster_data.insert(0, "segment_id", [f"S{i:04d}" for i in range(len(z))])
cluster_data["label_pembangkit"] = z
cluster_data.to_csv(out_dir / "data_sintesis_cluster.csv", index=False)
print("Perkerasan sintesis:", cluster_data.shape, "kelompok pembangkit:", pd.Series(z).nunique())


Perkerasan sintesis: (700, 6) kelompok pembangkit: 4


## Batas penggunaan
Data latihan ini bukan hasil uji balok, inventaris jembatan atau survei kondisi jalan. Seed 51505 membuat generator dapat diulang. Notebook analisis P05B membaca CSV dari folder kerja yang sama.